# Grafos ponderados: `step_cost`, costo uniforme y $A^*$

En el 8-puzzle y en las Torres de Hanoi **todos los movimientos cuestan lo mismo**, asi que
el costo acumulado $g(n)$ es simplemente la profundidad del nodo. En muchos problemas eso no
es cierto: viajar de una ciudad a otra cuesta lo que mida la carretera.

Para esos casos el nodo tiene el atributo **step_cost**: el costo del *unico* movimiento que
va del padre al hijo. La busqueda se encarga de acumularlo:

$$g(n) = g(\text{padre}) + \text{step\_cost}(n)$$

Usamos el mapa de Rumania de Russell & Norvig para ver la diferencia entre:

- **BFS**, que minimiza el *numero de pasos* (ignora los kilometros),
- **costo uniforme** (Dijkstra), que es $A^*$ con $h(n)=0$, y
- $A^*$ con la distancia en linea recta como heuristica.


In [1]:
import sys,os
baile_path=os.path.abspath(os.path.join("..","src"))
if not baile_path in sys.path:
    sys.path.append(baile_path)
import SimpleSearch as ss


## El mapa

Un diccionario `ciudad -> [(vecino, distancia), ...]`. El grafo es no dirigido, por eso
cada carretera aparece en las dos ciudades que conecta.


In [2]:
rumania={
 'Arad':      [('Zerind',75),('Sibiu',140),('Timisoara',118)],
 'Zerind':    [('Arad',75),('Oradea',71)],
 'Oradea':    [('Zerind',71),('Sibiu',151)],
 'Sibiu':     [('Arad',140),('Oradea',151),('Fagaras',99),('Rimnicu',80)],
 'Timisoara': [('Arad',118),('Lugoj',111)],
 'Lugoj':     [('Timisoara',111),('Mehadia',70)],
 'Mehadia':   [('Lugoj',70),('Dobreta',75)],
 'Dobreta':   [('Mehadia',75),('Craiova',120)],
 'Craiova':   [('Dobreta',120),('Rimnicu',146),('Pitesti',138)],
 'Rimnicu':   [('Sibiu',80),('Craiova',146),('Pitesti',97)],
 'Fagaras':   [('Sibiu',99),('Bucarest',211)],
 'Pitesti':   [('Rimnicu',97),('Craiova',138),('Bucarest',101)],
 'Bucarest':  [('Fagaras',211),('Pitesti',101),('Giurgiu',90)],
 'Giurgiu':   [('Bucarest',90)],
}


## Sucesores, meta y heuristica

Lo unico nuevo respecto a los notebooks anteriores es que al crear el nodo hijo le pasamos
`step_cost=distancia`. Si se omite, vale 1 y el comportamiento es el de siempre.


In [3]:
def sucesores(nodo):
    hijos=[]
    for vecino,distancia in rumania[nodo.state]:
        hijos.append(ss.node(vecino, parent=nodo, depth=nodo.depth+1,
                             op=f'{nodo.state} -> {vecino}',
                             step_cost=distancia)) # <- costo de esta carretera
    return hijos

def meta(*nodos):
    return nodos[0].state == nodos[1].state


La heuristica es la distancia **en linea recta** de cada ciudad a Bucarest. Nunca
sobreestima la distancia real por carretera (es *admisible*), que es la condicion que
necesita $A^*$ para garantizar la ruta optima.


In [4]:
linea_recta={'Arad':366,'Bucarest':0,'Craiova':160,'Dobreta':242,'Fagaras':176,
             'Giurgiu':77,'Lugoj':244,'Mehadia':241,'Oradea':380,'Pitesti':100,
             'Rimnicu':193,'Sibiu':253,'Timisoara':329,'Zerind':374}

def distancia_recta(nodo, meta_nodo):
    return linea_recta[nodo.state]


## Las tres busquedas

Mismo problema, misma interfaz; solo cambia la estrategia.


In [5]:
arad=ss.node('Arad', op='inicio')
bucarest=ss.node('Bucarest')

bfs=ss.TreeSearch(arad, sucesores, meta, goal_state=bucarest, strategy='bfs')
uniforme=ss.TreeSearch(arad, sucesores, meta, goal_state=bucarest, strategy='a*')
astar=ss.TreeSearch(arad, sucesores, meta, goal_state=bucarest, strategy='a*',
                    heuristic=distancia_recta)


In [6]:
for nombre,busqueda in [('BFS',bfs),('Costo uniforme',uniforme),('A*',astar)]:
    r=busqueda.find()
    ruta=' -> '.join(estado for estado,op,d in r.getPath())
    print('%-15s costo: %4d km  expandidos: %2d' %(nombre, r.cost, busqueda.iterations))
    print('%-15s %s\n' %('', ruta))


BFS             costo:  450 km  expandidos:  8
                Arad -> Sibiu -> Fagaras -> Bucarest

Costo uniforme  costo:  418 km  expandidos: 12
                Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucarest

A*              costo:  418 km  expandidos:  5
                Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucarest



## Que muestra el resultado

- **BFS** llega a Bucarest en 3 pasos (Arad, Sibiu, Fagaras, Bucarest) pero recorre 450 km.
  Es optimo en *numero de pasos*, no en *costo*: BFS solo garantiza optimalidad cuando todas
  las aristas cuestan igual.
- **Costo uniforme** encuentra la ruta de 418 km via Rimnicu y Pitesti, que es la mas barata
  aunque tenga un paso mas. Es $A^*$ con $h(n)=0$: ordena la frontera solo por $g(n)$.
- $A^*$ devuelve **la misma ruta optima** expandiendo menos ciudades, porque $f(n)=g(n)+h(n)$
  descarta pronto las direcciones que se alejan de Bucarest (Zerind, Timisoara, Oradea).

### Por que no se cierra un estado al generarlo

Bucarest entra a la frontera **dos veces**: primero por Fagaras con $g=450$ y despues por
Pitesti con $g=418$. Dos detalles del algoritmo hacen que gane la segunda:

1. La prueba de meta se aplica al **sacar** un nodo de la frontera, no al generarlo. Si se
   aplicara al generarlo, la busqueda terminaria con la ruta de 450 km.
2. `TreeSearch` guarda en el diccionario `reached` el mejor $g(n)$ conocido de cada estado y
   vuelve a insertar el nodo cuando llega uno mas barato. Marcar el estado como visitado la
   primera vez que aparece descartaria el camino de 418 km y $A^*$ dejaria de ser optimo.


In [7]:
via_fagaras=140+99+211        # Arad -> Sibiu -> Fagaras -> Bucarest
via_pitesti=140+80+97+101     # Arad -> Sibiu -> Rimnicu -> Pitesti -> Bucarest
print('Bucarest via Fagaras:', via_fagaras, 'km  (se genera primero)')
print('Bucarest via Pitesti:', via_pitesti, 'km  (se genera despues, pero es mas barata)')
print('g(Bucarest) que conserva la busqueda:', uniforme.reached['Bucarest'], 'km')


Bucarest via Fagaras: 450 km  (se genera primero)
Bucarest via Pitesti: 418 km  (se genera despues, pero es mas barata)
g(Bucarest) que conserva la busqueda: 418 km


In [8]:
# El mejor costo conocido por ciudad al terminar A*
for ciudad,g in sorted(astar.reached.items(), key=lambda kv: kv[1]):
    print('%-10s g = %4d km' %(ciudad,g))


Arad       g =    0 km
Zerind     g =   75 km
Timisoara  g =  118 km
Sibiu      g =  140 km
Rimnicu    g =  220 km
Fagaras    g =  239 km
Oradea     g =  291 km
Pitesti    g =  317 km
Craiova    g =  366 km
Bucarest   g =  418 km


## Ejercicios

1. Corre las tres busquedas desde **Timisoara** hasta **Giurgiu** y compara costo y nodos expandidos.
2. Que pasa con $A^*$ si multiplicas `linea_recta` por 3? Sigue siendo admisible la heuristica?
   Sigue siendo optima la ruta? Cuantos nodos expande?
3. Cambia `step_cost=distancia` por `step_cost=1`. Que estrategia da ahora la misma respuesta que BFS y por que?
